In [4]:
"""
Clean + dedupe a prompt-classification dataset, and reassign any
Summarization rows labeled "Easy" to Medium or Hard based on content.

HOW TO USE
----------
1. Edit the CONFIG section below:
   - INPUT_PATH: path to your CSV (or a folder of CSVs, see note below)
   - PROMPT_COL, TASK_TYPE_COL, DIFFICULTY_COL: your actual column names
2. Run:  python clean_dataset.py
3. Output: clean CSV written to OUTPUT_PATH, with a printed summary.

If you have MULTIPLE source CSVs to merge first, set INPUT_PATH to a list
of paths in the CONFIG section (see INPUT_PATHS).
"""

import re
import difflib
import pandas as pd

# ============================== CONFIG ==============================
INPUT_PATH = r"C:\Users\abdul\OneDrive\Documents\Dataset_propmts.csv"
# Leave empty to read INPUT_PATH. Add CSV paths here only when merging files.
INPUT_PATHS = []

OUTPUT_PATH = "dataset_clean.csv"

PROMPT_COL = "Prompt"          # column holding the actual prompt text
TASK_TYPE_COL = "Task Type"    # column holding the task category (e.g. "Summarization")
DIFFICULTY_COL = "Difficulty"  # column holding Easy/Medium/Hard
ID_COL = "ID"                  # column to renumber sequentially (created if missing)

SUMMARIZATION_LABEL = "Summarization"  # exact string used for that task_type
NEAR_DUP_THRESHOLD = 0.92              # similarity ratio above which two prompts count as near-duplicates
# ======================================================================


def load_data():
    if INPUT_PATHS:
        dfs = [pd.read_csv(p) for p in INPUT_PATHS]
        df = pd.concat(dfs, ignore_index=True)
    else:
        df = pd.read_csv(INPUT_PATH)
    return df


def normalize_text(s: str) -> str:
    s = str(s).strip().lower()
    s = re.sub(r"\s+", " ", s)
    s = re.sub(r"[^\w\s]", "", s)
    return s


def dedupe_exact(df: pd.DataFrame) -> pd.DataFrame:
    before = len(df)
    df["_norm"] = df[PROMPT_COL].apply(normalize_text)
    df = df.drop_duplicates(subset="_norm", keep="first").copy()
    removed = before - len(df)
    print(f"Exact-duplicate rows removed: {removed}")
    return df


def dedupe_near(df: pd.DataFrame) -> pd.DataFrame:
    """
    O(n^2) near-duplicate check using difflib similarity ratio.
    Fine for a few thousand rows; for much larger datasets swap in
    embeddings + cosine similarity instead.
    """
    texts = df["_norm"].tolist()
    idx = df.index.tolist()
    to_drop = set()

    for i in range(len(texts)):
        if idx[i] in to_drop:
            continue
        for j in range(i + 1, len(texts)):
            if idx[j] in to_drop:
                continue
            ratio = difflib.SequenceMatcher(None, texts[i], texts[j]).ratio()
            if ratio >= NEAR_DUP_THRESHOLD:
                to_drop.add(idx[j])  # keep the first occurrence, drop the later one

    if to_drop:
        df = df.drop(index=list(to_drop))
    print(f"Near-duplicate rows removed (ratio >= {NEAR_DUP_THRESHOLD}): {len(to_drop)}")
    return df


def classify_summarization_difficulty(prompt: str) -> str:
    """
    Heuristic reassignment for Summarization prompts that were
    originally labeled Easy. Never returns 'Easy'.

    Signals for Hard:
      - long/technical/legal source material (contract, report, paper,
        transcript, multi-page/word-count references)
      - multi-document or comparative summarization
      - domain jargon (legal, medical, financial, technical)

    Everything else defaults to Medium.
    """
    p = prompt.lower()

    hard_signals = [
        "legal contract", "contract", "research paper", "transcript",
        "multi-page", "multiple documents", "several documents",
        "10-page", "20-page", "lengthy", "financial report",
        "technical document", "whitepaper", "white paper",
        "medical record", "clinical", "patent", "compare and summarize",
        "summarize the following documents", "long document",
        "entire book", "full report", "quarterly report",
    ]

    if any(sig in p for sig in hard_signals):
        return "Hard"
    return "Medium"


def fix_summarization_labels(df: pd.DataFrame) -> pd.DataFrame:
    mask = (
        (df[TASK_TYPE_COL] == SUMMARIZATION_LABEL)
        & (df[DIFFICULTY_COL].str.lower() == "easy")
    )
    n_fixed = mask.sum()

    df.loc[mask, DIFFICULTY_COL] = df.loc[mask, PROMPT_COL].apply(
        classify_summarization_difficulty
    )

    print(f"Summarization rows reassigned from Easy: {n_fixed}")
    return df


def renumber_ids(df: pd.DataFrame) -> pd.DataFrame:
    df = df.reset_index(drop=True)
    df[ID_COL] = range(1, len(df) + 1)
    # move id column to front
    cols = [ID_COL] + [c for c in df.columns if c != ID_COL]
    return df[cols]


def main():
    df = load_data()
    print(f"Loaded {len(df)} rows.")

    df = dedupe_exact(df)
    df = dedupe_near(df)
    df = fix_summarization_labels(df)

    df = df.drop(columns=["_norm"])
    df = renumber_ids(df)

    df.to_csv(OUTPUT_PATH, index=False)

    print("\n--- Summary ---")
    print(f"Final row count: {len(df)}")
    if DIFFICULTY_COL in df.columns:
        print(df[DIFFICULTY_COL].value_counts())
    if TASK_TYPE_COL in df.columns and SUMMARIZATION_LABEL in df[TASK_TYPE_COL].unique():
        print("\nSummarization difficulty breakdown:")
        print(
            df[df[TASK_TYPE_COL] == SUMMARIZATION_LABEL][DIFFICULTY_COL]
            .value_counts()
        )
    print(f"\nSaved clean dataset to: {OUTPUT_PATH}")


if __name__ == "__main__":
    main()

Loaded 1300 rows.
Exact-duplicate rows removed: 0
Near-duplicate rows removed (ratio >= 0.92): 138
Summarization rows reassigned from Easy: 0

--- Summary ---
Final row count: 1162
Difficulty
Easy      559
Hard      344
Medium    259
Name: count, dtype: int64

Summarization difficulty breakdown:
Difficulty
Medium    30
Hard      11
Name: count, dtype: int64

Saved clean dataset to: dataset_clean.csv
